In [ ]:
import os
import re
import pypdf
import ollama
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Config
PDF_PATH = "de.pdf" 
MODEL = "llama3.2" 
OUTPUT_MD = "de_interview_prep.md"

chapters_list = [
    """Chapter 1. Data Engineering""",
    """Chapter 2. The Data
Engineering Lifecycle""",
    """Chapter 3. Designing Good
Data Architecture""",
    """Chapter 4. Choosing
Technologies Across the Data
Engineering Lifecycle""",
"""Chapter 5. Data Generation in
Source Systems""",
"""Chapter 6. Storage""",
"""Chapter 7. Ingestion""",
"""Chapter 8. Queries, Modeling, and
Transformation""",
"""Chapter 9. Serving Data for
Analytics, Machine Learning,
and Reverse ETL""",
"""Chapter 10. Security and
Privacy""",
"""Chapter 11. The Future of Data
Engineering"""
]

def get_chapter_hierarchy(pdf_path, chapters):
    reader = pypdf.PdfReader(pdf_path)
    hierarchy = []
    
    # We look for "Chapter X" and "X.X" patterns to avoid skipping subsections
    section_pattern = re.compile(r'^(\d+\.\d+)\s+(.*)') 

    for i, title in enumerate(chapters):
        start_page = -1
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text()
            if title.lower() in text.lower():
                start_page = page_num
                break
        
        if start_page != -1:
            hierarchy.append({"title": title, "page": start_page, "type": "chapter"})

    return hierarchy

def get_chapter_ranges(pdf_path, chapters):
    """Finds the start and end page for each chapter title."""
    reader = pypdf.PdfReader(pdf_path)
    chapter_pages = []
    
    print("Mapping chapters to page numbers...")
    for title in chapters:
        found_page = -1
        # Search for the title string in the PDF text
        for i, page in enumerate(reader.pages):
            if title.lower() in page.extract_text().lower():
                found_page = i 
                break
        chapter_pages.append({"title": title, "start": found_page})

    # Calculate end pages based on the next chapter's start
    for i in range(len(chapter_pages)):
        if chapter_pages[i]["start"] == -1:
            continue
        if i + 1 < len(chapter_pages) and chapter_pages[i+1]["start"] != -1:
            chapter_pages[i]["end"] = chapter_pages[i+1]["start"]
        else:
            chapter_pages[i]["end"] = len(reader.pages)
            
    return [c for c in chapter_pages if c["start"] != -1]

def extract_text_from_range(pdf_path, start_page, end_page):
    """Extracts text only within the specified page boundaries."""
    text = ""
    try:
        reader = pypdf.PdfReader(pdf_path)
        for i in range(start_page, end_page):
            extracted = reader.pages[i].extract_text()
            if extracted:
                text += extracted + "\n"
    except Exception as e:
        print(f"Error reading range {start_page}-{end_page}: {e}")
    return text

def _extract_text(pdf_path):
    """Fixed: Uses pypdf correctly and handles empty pages."""
    text = ""
    try:
        reader = pypdf.PdfReader(pdf_path)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted:
                text += extracted + "\n"
    except Exception as e:
        print(f"Error reading PDF: {e}")
    return text

def _detect_chapters(text):
    """Fixed: Correctly pairs detected titles with their following content."""
    # Using a capturing group keeps the titles in the split list
    parts = re.split(r'(Chapter \d+|CHAPTER \d+|Section \d+)', text)
    
    chapters = []
    # Skip index 0 (intro text) and step by 2 to get [Title, Content] pairs
    for i in range(1, len(parts), 2):
        title = parts[i]
        content = parts[i+1] if i+1 < len(parts) else ""
        chapters.append(f"{title}\n{content}")
    
    return chapters[:20]

def summarize_chapter(chapter_text, chapter_num, chapter_title):
    prompt_template = """
    You are a Staff/Principal Data Engineering architect and interviewer. Write like someone who has built and operated production data platforms.

    Your task is to produce a deeply explanatory, interview‑grade summary of the provided text from "{chapter_title}".

    ### CRITICAL INSTRUCTION: NO-SKIP AUDIT (Conditional Inclusion)
    1. Identify EVERY sub-section, technical concept, and architectural pattern in the text.
    2. For EACH distinct topic found, you MUST provide a detailed breakdown.
    3. **IF APPLICABLE ONLY:** Only include the following headers if the text provides enough detail. Do not force empty sections.

    Structure your output EXACTLY as follows for EVERY sub-topic identified:

    ## Section: [Original Name from PDF]

    ### Core Concepts and Their Rationale (If Applicable)
    - **What it is:** (1–2 lines, no textbook definitions)
    - **Why it exists:** (Real engineering pain it solved)
    - **What breaks without it:** (Data quality, reliability, cost, governance)
    - **Enterprise example:** {{industry}} + {{scale}} + {{constraint}} + {{failure mode avoided}} + {{measurable outcome}}.
    - **Anti-example:** (When it becomes overkill or harmful)

    ### Architectures and Design Choices (If Applicable)
    - **Why it is chosen:** (Over simpler designs)
    - **Trade-offs:** (Explicit: cost vs complexity, latency vs correctness)
    - **Enterprise example:** (With {{scale}} + {{SLA}} + {{compliance/governance}} angle)
    - **Failure modes:** (What goes wrong in real orgs)
    - **When NOT to use it:** (Team maturity, budget, use case mismatch)
    - **Interview defense:** (How you justify it to an interviewer)

    ### Tools and Platforms (If Applicable)
    - **Why it exists:** (Operational pain it abstracts)
    - **Enterprise example:** (Migration, modernization, multi-team platform)
    - **Why it beats alternatives:** (And where it loses)
    - **Cost / governance risks:** (Runaway compute, poor lineage, secrets, RBAC)
    - **Common misuse:** (That causes outages or bad data)

    ### Best Practices Explained (If Applicable)
    - **Why it’s non-negotiable at scale**
    - **Enterprise example:** (What incident it prevented or fixed)
    - **Failure mode if ignored:** (Silent corruption, SLA miss, audit failure)
    - **How seniors enforce it:** (Guardrails, templates, policies, CI/CD)

    ### Interview-Level Insights (Include for every section)
    - 3–5 high-signal “WHY” interview questions specifically for this sub-topic.
    - Always provide eloquent answers referencing trade-offs and constraints.
    - Red flags interviewers look for.

    ---
    TEXT TO ANALYZE:
    {chunk_text}

    ### FINAL REMINDER:
    - If a section is purely conceptual, focus on the Rationale and Best Practices.
    - If it is technical, focus on Architecture and Tools.
    - DO NOT summarize the whole text as one; treat every sub-topic as a separate entry.
    - Interview-Level Insights MUST HAVE specific questions and answers for each sub-topic, not just generic ones.
    """

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=3000, 
        chunk_overlap=300,
        separators=["\n### ", "\n## ", "\n\n", "\n"]
    )
    chunks = splitter.split_text(chapter_text)
    
    chapter_summary = ""
    for i, chunk in enumerate(chunks):
        # We only pass chapter_title and chunk_text. 
        # Double braces {{ }} are for Ollama to handle.
        current_prompt = prompt_template.format(
            chapter_title=chapter_title,
            chunk_text=chunk
        )
        
        response = ollama.generate(model=MODEL, prompt=current_prompt)
        chapter_summary += response['response'] + "\n\n"
        print(f"Processed chunk {i+1}/{len(chunks)} for {chapter_title}")

    return chapter_summary


def generate_mocks(summary):
    """Maintains your exact mock interview prompt."""
    # YOUR ORIGINAL PROMPT
    prompt = f"""
    Based on this chapter summary, generate EXACTLY 3 mock Data Engineering interview questions (behavioral/technical). 
    For each: Question, then curated acceptable answer (concise, eloquent, 150-250 words).
    
    Format as:
    ### Mock Question 1
    **Q:** [Question]
    **A:** [Answer]
    
    Output ONLY markdown.
    """
    # Truncate to ensure the prompt doesn't exceed LLM limits
    response = ollama.generate(model=MODEL, prompt=prompt.replace("this chapter summary", summary[:8000]))
    return response['response']

# Main execution
if __name__ == "__main__":
    if not os.path.exists(PDF_PATH):
        print(f"File not found: {PDF_PATH}")
    else:
        # 1. Map the chapters to pages
        ranges = get_chapter_ranges(PDF_PATH, chapters_list)
        md_content = "# Data Engineering Interview Prep\n\n"

        # 2. Process each range
        for i, item in enumerate(ranges, 1):
            print(f"Processing {item['title']} (Pages {item['start']} to {item['end']})...")
            
            # Extract only the relevant text
            chapter_text = extract_text_from_range(PDF_PATH, item['start'], item['end'])
            
            if not chapter_text.strip():
                continue

            # 3. Summarize and Mock
            chap_summary = summarize_chapter(chapter_text, i, item['title'])
            mocks = generate_mocks(chap_summary)
            print(chap_summary)

            md_content += f"{chap_summary}\n\n{mocks}\n\n---\n\n"
            
            # Incremental save
            with open(OUTPUT_MD, 'w', encoding='utf-8') as f:
                f.write(md_content)

        print(f"Done! Created: {OUTPUT_MD}")

Mapping chapters to page numbers...
Processing Chapter 1. Data Engineering (Pages 17 to 59)...
Processed chunk 1/25 for Chapter 1. Data Engineering
Processed chunk 2/25 for Chapter 1. Data Engineering
Processed chunk 3/25 for Chapter 1. Data Engineering
Processed chunk 4/25 for Chapter 1. Data Engineering
Processed chunk 5/25 for Chapter 1. Data Engineering
Processed chunk 6/25 for Chapter 1. Data Engineering
Processed chunk 7/25 for Chapter 1. Data Engineering
Processed chunk 8/25 for Chapter 1. Data Engineering
Processed chunk 9/25 for Chapter 1. Data Engineering
Processed chunk 10/25 for Chapter 1. Data Engineering
Processed chunk 11/25 for Chapter 1. Data Engineering
Processed chunk 12/25 for Chapter 1. Data Engineering
Processed chunk 13/25 for Chapter 1. Data Engineering
Processed chunk 14/25 for Chapter 1. Data Engineering
Processed chunk 15/25 for Chapter 1. Data Engineering
Processed chunk 16/25 for Chapter 1. Data Engineering
Processed chunk 17/25 for Chapter 1. Data Engineer